# Tukey Median Polish and Cosine Scoring

This notebook illustrates the **Tukey median polish** algorithm as used in Osprey, and
how it produces three discriminating features:

| Feature | Description |
|---------|-------------|
| `median_polish_cosine` | Cosine similarity between the polished elution profile and the library fragment pattern |
| `median_polish_rsquared` | R² of the additive model fit |
| `median_polish_residual_ratio` | Fraction of total variance explained by the additive model |

## Intuition

The fragment XIC matrix has rows = fragments and columns = scans. For a **true peptide
peak** this matrix has a very clean structure:

$$x_{fs} \approx \mu + \alpha_f + \beta_s$$

where $\mu$ is the overall level, $\alpha_f$ is the fragment-specific offset (encodes
the library pattern), and $\beta_s$ is the elution profile shared across all fragments
(the peak shape). The **residuals** after removing this additive structure are small.

For an **interference peak**, fragments do NOT share the same elution profile — the
additive model fits poorly, residuals are large, and the extracted $\beta_s$ ("elution
profile") does not match the library fragment pattern.

Osprey works in **log-intensity space** so that the multiplicative fragment pattern
becomes additive, matching the standard Tukey model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

rng = np.random.default_rng(13)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def gaussian(rt, apex, h, s):
    return h * np.exp(-0.5 * ((rt - apex) / s) ** 2)


def nanmedian_1d(arr):
    vals = [v for v in arr if np.isfinite(v)]
    return np.median(vals) if vals else np.nan

## 1. Simulate the Fragment XIC Matrix

We create a 6×20 matrix (6 fragments × 20 scans within the peak window) in three
scenarios:

- **Scenario A**: Clean peptide peak — all fragments share the same Gaussian elution  
- **Scenario B**: Mild interference — one fragment has a shifted/broadened peak  
- **Scenario C**: Strong interference — fragments have largely unrelated elution shapes

In [ ]:
n_frags = 6
n_scans = 24

# Scan axis (arbitrary units within the peak window)
t = np.linspace(-3, 3, n_scans)   # in units of σ

# Library fragment relative intensities (arbitrary, will become row offsets in log space)
lib_ints = np.array([100, 75, 55, 40, 28, 18], dtype=float)
frag_labels = [f'b{i+2}²⁺' if i < 3 else f'y{i+1}⁺' for i in range(n_frags)]
frag_colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Shared Gaussian elution profile
elution = np.exp(-0.5 * t**2)   # unit Gaussian

noise_frac = 0.06   # 6% multiplicative noise

# ── Scenario A: clean peptide peak ────────────────────────────────────────
mat_A = np.outer(lib_ints, elution)   # pure rank-1 structure
mat_A *= rng.lognormal(0, noise_frac, mat_A.shape)   # multiplicative noise
mat_A = np.clip(mat_A, 0.5, None)    # no zeros

# ── Scenario B: mild interference on fragment 0 ────────────────────────────
mat_B = mat_A.copy()
# Fragment 0 gets an extra bump at t=+1 (co-eluting peptide)
mat_B[0, :] += lib_ints[0] * 0.5 * np.exp(-0.5 * ((t - 1.2) / 0.6) ** 2)
mat_B *= rng.lognormal(0, noise_frac, mat_B.shape)
mat_B = np.clip(mat_B, 0.5, None)

# ── Scenario C: strong interference — fragments mostly unrelated ──────────
mat_C = np.zeros((n_frags, n_scans))
for i in range(n_frags):
    noise_apex = rng.uniform(-1.5, 1.5)   # each fragment peaks at a different RT
    mat_C[i, :] = lib_ints[i] * np.exp(-0.5 * ((t - noise_apex) / rng.uniform(0.5, 1.5)) ** 2)
mat_C *= rng.lognormal(0, 0.12, mat_C.shape)
mat_C = np.clip(mat_C, 0.5, None)

scenarios = [
    ('A — Clean Peptide Peak', mat_A),
    ('B — Mild Interference (1 fragment)', mat_B),
    ('C — Strong Interference', mat_C),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, mat) in zip(axes, scenarios):
    for i, (row, color, label) in enumerate(zip(mat, frag_colors, frag_labels)):
        ax.plot(t, row, color=color, lw=1.5, label=label)
    ax.set_title(f'Scenario {title}', fontsize=10)
    ax.set_xlabel('Scan (σ units)')
    if ax is axes[0]:
        ax.set_ylabel('Intensity')
        ax.legend(fontsize=7.5, ncol=2, frameon=False)

fig.suptitle('Fragment XIC Matrix — Three Scenarios', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Median Polish Algorithm

Tukey median polish iteratively removes **row medians** and **column medians** from the
residual matrix until convergence. It is a robust (median-based) decomposition that
produces:

- **Overall** $\mu$: grand median level
- **Row effects** $\alpha_f$: per-fragment offset (encodes library pattern in log space)
- **Column effects** $\beta_s$: per-scan offset (encodes elution profile / peak shape)
- **Residuals** $r_{fs}$: unexplained variation (interference, noise)

$$\ln(x_{fs}) = \mu + \alpha_f + \beta_s + r_{fs}$$

In [ ]:
def tukey_median_polish(mat, max_iter=50, tol=1e-4):
    """
    Tukey median polish in log-intensity space.
    Returns dict with keys: overall, row_effects, col_effects, residuals.
    """
    n_rows, n_cols = mat.shape
    
    # Log transform (NaN for zeros)
    residuals = np.where(mat > 0, np.log(mat.astype(float)), np.nan)
    
    overall     = 0.0
    row_effects = np.zeros(n_rows)
    col_effects = np.zeros(n_cols)
    
    history = []
    
    for iteration in range(max_iter):
        old = residuals.copy()
        
        # Row sweep
        row_meds = np.array([nanmedian_1d(residuals[r]) for r in range(n_rows)])
        for r in range(n_rows):
            if np.isfinite(row_meds[r]):
                residuals[r, :] -= row_meds[r]
        med_of_row_meds = nanmedian_1d(row_meds)
        if np.isfinite(med_of_row_meds):
            row_effects += row_meds - med_of_row_meds
            overall += med_of_row_meds
        
        # Column sweep
        col_meds = np.array([nanmedian_1d(residuals[:, c]) for c in range(n_cols)])
        for c in range(n_cols):
            if np.isfinite(col_meds[c]):
                residuals[:, c] -= col_meds[c]
        med_of_col_meds = nanmedian_1d(col_meds)
        if np.isfinite(med_of_col_meds):
            col_effects += col_meds - med_of_col_meds
            overall += med_of_col_meds
        
        # Convergence
        diff = np.nanmax(np.abs(residuals - old))
        history.append({'iter': iteration+1, 'max_change': diff, 'n_positive_col': int((col_effects > 0).sum())})
        if diff < tol:
            break
    
    return {
        'overall': overall,
        'row_effects': row_effects,
        'col_effects': col_effects,
        'residuals': residuals,
        'history': history,
        'n_iter': len(history),
    }


results = {name: tukey_median_polish(mat) for name, mat in scenarios}

for name, mat in scenarios:
    r = results[name]
    max_resid = np.nanmax(np.abs(r['residuals']))
    print(f'  {name[:5]}  converged in {r["n_iter"]:>2d} iterations, '
          f'max residual = {max_resid:.4f},  overall = {r["overall"]:.3f}')

## 3. Visualise the Decomposition

For each scenario we show the four components of the additive decomposition as heatmaps.
The **column effects** ($\beta_s$) encode the consensus elution profile — this is what
gets compared to the library to compute `median_polish_cosine`.

In [ ]:
def plot_decomposition(mat, result, title, fig, outer_gs_pos):
    """Plot raw matrix, fitted additive structure, and residuals side by side."""
    n_rows, n_cols = mat.shape
    
    r = result
    # Reconstruct fitted log-matrix: overall + row_effects[:, None] + col_effects[None, :]
    fitted_log = (r['overall']
                  + r['row_effects'][:, None]
                  + r['col_effects'][None, :])
    log_mat = np.where(mat > 0, np.log(mat), np.nan)
    
    inner = outer_gs_pos.subgridspec(1, 4, wspace=0.3)
    axes = [fig.add_subplot(inner[0, i]) for i in range(4)]
    
    tick_rts = np.arange(0, n_cols, 6)
    scan_labels = [f'{t[i]:.1f}σ' for i in tick_rts]
    
    # Raw log-matrix
    im0 = axes[0].imshow(log_mat, aspect='auto', cmap='YlOrRd',
                         vmin=np.nanmin(log_mat), vmax=np.nanmax(log_mat))
    axes[0].set_title('Raw (log intensity)', fontsize=8.5)
    plt.colorbar(im0, ax=axes[0], shrink=0.6, pad=0.02)
    
    # Fitted (additive model)
    im1 = axes[1].imshow(fitted_log, aspect='auto', cmap='YlOrRd',
                         vmin=np.nanmin(log_mat), vmax=np.nanmax(log_mat))
    axes[1].set_title('Fitted: μ + α_f + β_s', fontsize=8.5)
    plt.colorbar(im1, ax=axes[1], shrink=0.6, pad=0.02)
    
    # Residuals
    res = r['residuals']
    vlim = max(0.2, np.nanpercentile(np.abs(res), 95))
    norm = TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
    im2 = axes[2].imshow(res, aspect='auto', cmap='RdBu_r', norm=norm)
    axes[2].set_title('Residuals', fontsize=8.5)
    plt.colorbar(im2, ax=axes[2], shrink=0.6, pad=0.02)
    
    # Column effects (elution profile) + row effects (fragment pattern)
    ax4 = axes[3]
    ax4_r = ax4.twinx()
    col_eff = r['col_effects']
    row_eff = r['row_effects']
    
    ax4.plot(t, col_eff, color='steelblue', lw=2, label='β (elution)')
    ax4.axhline(0, color='gray', lw=0.6, ls='--')
    ax4.set_xlabel('Scan', fontsize=8)
    ax4.set_ylabel('β_s (col effect)', fontsize=8, color='steelblue')
    ax4.tick_params(axis='y', labelsize=7, labelcolor='steelblue')
    
    # Row effects as bar on the right y-axis
    x_pos = np.arange(n_rows)
    ax4_r.barh(x_pos, row_eff, height=0.6, alpha=0.4, color=frag_colors, label='α (fragment)')
    ax4_r.invert_yaxis()
    ax4_r.set_yticks(x_pos)
    ax4_r.set_yticklabels(frag_labels, fontsize=7)
    ax4_r.set_ylabel('α_f (row effects)', fontsize=8, color='#666')
    ax4_r.tick_params(axis='y', labelsize=7)
    ax4.set_title('β (elution) + α (fragment\nrow effects)', fontsize=8.5)
    
    for ax in axes[:3]:
        ax.set_yticks(range(n_rows))
        ax.set_yticklabels(frag_labels, fontsize=7)
        ax.set_xticks(tick_rts)
        ax.set_xticklabels(scan_labels, fontsize=7)
    
    # Scenario title on leftmost panel
    axes[0].set_ylabel(title, fontsize=9, fontweight='bold', labelpad=40, color='#1a1a2e')


fig = plt.figure(figsize=(18, 15))
outer_gs = gridspec.GridSpec(3, 1, figure=fig, hspace=0.50)

for i, (name, mat) in enumerate(scenarios):
    plot_decomposition(mat, results[name], name, fig, outer_gs[i])

fig.suptitle('Tukey Median Polish Decomposition: ln(X) = μ + α_f + β_s + r\n'
             '(columns = scans, rows = fragments)',
             fontsize=13, fontweight='bold', y=0.99)
plt.savefig('median_polish_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: median_polish_decomposition.png')

## 4. The Three Osprey Median Polish Features

### `median_polish_cosine`

The column effects ($\beta_s$, the consensus elution profile) are compared to the library
fragment intensities. Specifically, the **exponential** of the column effects is taken
(back-transforming from log space) to get a relative intensity profile, then its cosine
similarity with the library intensities is computed.

A **high cosine** means the consensus elution profile correctly orders the fragment
intensities — exactly what is expected if fragments belong to the same peptide.
With interference, the elution profile extracted is a mixture signal and the cosine
will be lower.

> **Note**: this cosine captures a different aspect than `elution_weighted_cosine`. The
> median polish cosine works on the *extracted consensus profile* (a rank-1 approximation
> that maximally agrees with all fragments), while `elution_weighted_cosine` scores each
> observed spectrum against the library at every scan independently.

In [ ]:
def lib_cosine_from_col_effects(col_effects, row_effects, lib_ints):
    """
    Osprey's median_polish_libcosine:
    At each scan, reconstruct predicted fragment intensities = exp(overall + alpha_f + beta_s)
    → compare to library with sqrt-preprocessed cosine.
    Return the cosine at the apex scan (argmax of col_effects).
    """
    apex_scan = np.argmax(col_effects)
    # Predicted relative intensities at apex scan (ignore overall constant — cancel in cosine)
    pred = np.exp(row_effects + col_effects[apex_scan])   # shape (n_frags,)
    lib = np.sqrt(lib_ints)
    obs = np.sqrt(np.clip(pred, 0, None))
    nl = np.linalg.norm(lib)
    no = np.linalg.norm(obs)
    if nl < 1e-10 or no < 1e-10:
        return 0.0
    return float(np.dot(lib/nl, obs/no))


def mp_rsquared(mat, result):
    """R² of the additive model."""
    log_mat = np.where(mat > 0, np.log(mat.astype(float)), np.nan)
    fitted = (result['overall']
              + result['row_effects'][:, None]
              + result['col_effects'][None, :])
    valid = np.isfinite(log_mat)
    ss_res = np.nansum((log_mat - fitted) ** 2)
    mean_all = np.nanmean(log_mat)
    ss_tot = np.nansum((log_mat - mean_all) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0


def mp_residual_ratio(result):
    """Ratio of row/col signal to residual magnitude."""
    signal = np.std(result['col_effects']) + np.std(result['row_effects'])
    noise  = np.nanstd(result['residuals'])
    return signal / (signal + noise + 1e-10)


print('Feature                       Scenario A       Scenario B       Scenario C')
print('─' * 80)

feature_vals = {name: {} for name, _ in scenarios}
for name, mat in scenarios:
    r = results[name]
    cos  = lib_cosine_from_col_effects(r['col_effects'], r['row_effects'], lib_ints)
    r2   = mp_rsquared(mat, r)
    ratio = mp_residual_ratio(r)
    feature_vals[name] = {'cosine': cos, 'r2': r2, 'ratio': ratio}

metrics = [('median_polish_cosine', 'cosine'),
           ('median_polish_R²', 'r2'),
           ('median_polish_residual_ratio', 'ratio')]
for label, key in metrics:
    vals = [feature_vals[n][key] for n, _ in scenarios]
    print(f'  {label:<30}  {vals[0]:>8.4f}  {vals[1]:>15.4f}  {vals[2]:>15.4f}')

## 5. Summary Figure — Features vs Scenario

In [ ]:
scenario_short = ['A\n(clean)', 'B\n(mild\ninterference)', 'C\n(strong\ninterference)']
bar_colors = ['#4dac26', '#d4701c', '#d62728']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left panel: per-scenario feature bar chart ─────────────────────────────
ax = axes[0]
x = np.arange(3)
w = 0.27

cosines  = [feature_vals[n]['cosine'] for n, _ in scenarios]
r2s      = [feature_vals[n]['r2']     for n, _ in scenarios]
ratios   = [feature_vals[n]['ratio']  for n, _ in scenarios]

ax.bar(x - w, cosines, w, label='mp_cosine', color='#2166ac', alpha=0.85)
ax.bar(x,     r2s,     w, label='mp_R²', color='#4dac26', alpha=0.85)
ax.bar(x + w, ratios,  w, label='mp_residual_ratio', color='#d62728', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(scenario_short, fontsize=10)
ax.set_ylabel('Feature value')
ax.set_ylim(0, 1.15)
ax.axhline(1.0, color='gray', lw=0.8, ls='--')
ax.legend(fontsize=9, frameon=False)
ax.set_title('Median Polish Features by Scenario',
             fontweight='bold', fontsize=11)

# Annotate bars with values
for bars in [ax.containers[0], ax.containers[1], ax.containers[2]]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=7.5)

# ── Right panel: elution profiles extracted per scenario ───────────────────
ax2 = axes[1]
line_styles = ['-', '--', ':']
line_colors = ['#4dac26', '#d4701c', '#d62728']
scen_labels = ['A — Clean', 'B — Mild interference', 'C — Strong interference']

for (name, mat), ls, lc, sl in zip(scenarios, line_styles, line_colors, scen_labels):
    col_eff = results[name]['col_effects']
    # Normalise to [0, 1] for comparison
    col_norm = col_eff - col_eff.min()
    if col_norm.max() > 0:
        col_norm /= col_norm.max()
    ax2.plot(t, col_norm, ls=ls, color=lc, lw=2.2, label=sl)

# Overlay the true elution profile (normalised)
ax2.plot(t, elution, color='black', lw=1.2, alpha=0.5, ls='-', label='True peptide profile')

ax2.set_xlabel('Scan (σ units)')
ax2.set_ylabel('Normalised column effect (β_s)')
ax2.set_title('Extracted Elution Profile (β_s) per Scenario\n'
              'Agreement with true profile → high mp_cosine',
              fontweight='bold', fontsize=10)
ax2.legend(fontsize=8.5, frameon=False)

plt.tight_layout()
plt.savefig('median_polish_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: median_polish_features.png')

## 6. Convergence Illustration

The median polish typically converges in 5–20 iterations. Here we trace the maximum
residual change across iterations to show the convergence behaviour for all three
scenarios.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for (name, mat), lc in zip(scenarios, ['#4dac26', '#d4701c', '#d62728']):
    hist = results[name]['history']
    iters  = [h['iter'] for h in hist]
    deltas = [h['max_change'] for h in hist]
    ax.semilogy(iters, deltas, '-o', color=lc, lw=2, ms=5, label=name[:5].strip())

ax.axhline(1e-4, color='gray', lw=1, ls='--', label='Convergence threshold (1e-4)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Max residual change (log scale)')
ax.set_title('Median Polish Convergence', fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print('The clean scenario (A) converges faster; interference slows convergence.')